# Flowline Discharge Debug Notebook

This notebook fixes and validates flowline discharge display by:
- loading flowlines and monthly discharge sources,
- normalizing/joining keys,
- computing Jan 2018 and annual metrics,
- rendering clickable flowlines with discharge popups,
- reporting unmatched diagnostics.

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import folium
import numpy as np

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'inputData').exists():
    BASE_DIR = Path(r'C:/Users/mu3575/Documents/WAM')

FLOWLINE_PATH = BASE_DIR / 'inputData' / 'flowlines' / 'Brazos_Flowline.shp'
USGS_PATH = BASE_DIR / 'inputData' / 'inputs' / 'monthly_wide_acft.csv'
WAM_PATH = BASE_DIR / 'inputData' / 'inputs' / 'monthly_wide_acft_from_hecdss.csv'
ROUTED_PATH = BASE_DIR / 'output' / 'brazos' / 'modeled_monthly_comid_flows.csv'
MAP_OUTPUT_PATH = BASE_DIR / 'output' / 'usgs_wam_flowlines_texas_map.html'
UNMATCHED_OUTPUT_PATH = BASE_DIR / 'output' / 'unmatched_flowlines_jan2018.csv'

print('BASE_DIR:', BASE_DIR)
print('FLOWLINE_PATH exists:', FLOWLINE_PATH.exists())
print('USGS_PATH exists:', USGS_PATH.exists())
print('WAM_PATH exists:', WAM_PATH.exists())
print('ROUTED_PATH exists:', ROUTED_PATH.exists())

## 1. Verify Cell 6 Inputs and Runtime State

In [ ]:
flow_preview = gpd.read_file(FLOWLINE_PATH, rows=5)
routed_preview = pd.read_csv(ROUTED_PATH, nrows=5)

print('Flowline columns:', list(flow_preview.columns))
print('Flowline CRS:', flow_preview.crs)
print('Routed columns:', list(routed_preview.columns))
print('Flowline preview rows:', len(flow_preview))
print('Routed preview rows:', len(routed_preview))
display(flow_preview.head(2))
display(routed_preview.head(2))

## 2. Load Flowline Geometry and Monthly Discharge Sources

In [ ]:
flow_gdf = gpd.read_file(FLOWLINE_PATH).to_crs(4326)
flow_gdf = flow_gdf[flow_gdf.geometry.notna() & ~flow_gdf.geometry.is_empty].copy()

flow_keep = [c for c in ['COMID', 'REACHCODE', 'GNIS_NAME', 'LENGTHKM', 'geometry'] if c in flow_gdf.columns]
flow_gdf = flow_gdf[flow_keep].copy()

routed_df = pd.read_csv(ROUTED_PATH)
usgs_df = pd.read_csv(USGS_PATH, dtype=str)
wam_df = pd.read_csv(WAM_PATH, dtype=str)

print('Flowline rows:', len(flow_gdf))
print('Routed rows:', len(routed_df))
print('USGS rows:', len(usgs_df))
print('WAM rows:', len(wam_df))

## 3. Normalize Join Keys for Flowlines and Points

In [ ]:
def norm_key(series):
    return series.astype(str).str.strip().str.upper()

flow_gdf['join_comid'] = norm_key(pd.to_numeric(flow_gdf['COMID'], errors='coerce').fillna(-1).astype('int64'))

if 'brazos_comid' in routed_df.columns:
    routed_df['join_comid'] = norm_key(pd.to_numeric(routed_df['brazos_comid'], errors='coerce').fillna(-1).astype('int64'))
elif 'COMID' in routed_df.columns:
    routed_df['join_comid'] = norm_key(pd.to_numeric(routed_df['COMID'], errors='coerce').fillna(-1).astype('int64'))
else:
    raise ValueError('Routed CSV missing brazos_comid/COMID for join.')

print('Distinct flowline keys:', flow_gdf['join_comid'].nunique())
print('Distinct routed keys:', routed_df['join_comid'].nunique())

## 4. Compute Flowline Discharge Metrics for Mapping

In [ ]:
routed_df['year'] = pd.to_numeric(routed_df.get('year'), errors='coerce')
routed_df['month'] = pd.to_numeric(routed_df.get('month'), errors='coerce')
routed_df['flow_cms'] = pd.to_numeric(routed_df.get('flow_cms'), errors='coerce')
routed_df['flow_acft'] = pd.to_numeric(routed_df.get('flow_acft'), errors='coerce')

jan_2018 = routed_df[(routed_df['year'] == 2018) & (routed_df['month'] == 1)].copy()
annual_2018 = routed_df[routed_df['year'] == 2018].groupby('join_comid', as_index=False)['flow_acft'].sum()
annual_2018 = annual_2018.rename(columns={'flow_acft': 'discharge_annual_2018'})

jan_metric = jan_2018.groupby('join_comid', as_index=False).agg(
    discharge_jan_2018=('flow_acft', 'sum'),
    flow_cms_jan_2018=('flow_cms', 'sum')
)

metrics_df = jan_metric.merge(annual_2018, on='join_comid', how='left')
print('Metrics rows:', len(metrics_df))
display(metrics_df.head(5))

## 5. Join Discharge Back to Flowlines

In [ ]:
flow_map_gdf = flow_gdf.merge(metrics_df, on='join_comid', how='left')
flow_map_gdf['matched'] = flow_map_gdf['discharge_jan_2018'].notna()
flow_map_gdf['discharge_jan_2018'] = flow_map_gdf['discharge_jan_2018'].fillna(0.0)
flow_map_gdf['flow_cms_jan_2018'] = flow_map_gdf['flow_cms_jan_2018'].fillna(0.0)
flow_map_gdf['discharge_annual_2018'] = flow_map_gdf['discharge_annual_2018'].fillna(0.0)

flow_map_gdf['flow_cms_txt'] = flow_map_gdf['flow_cms_jan_2018'].map(lambda v: f'{v:,.3f}')
flow_map_gdf['flow_acft_txt'] = flow_map_gdf['discharge_jan_2018'].map(lambda v: f'{v:,.3f}')

print('Flowlines total:', len(flow_map_gdf))
print('Matched flowlines:', int(flow_map_gdf['matched'].sum()))

## 6. Render Folium Flowlines with Discharge Tooltip/Popup

In [ ]:
m = folium.Map(location=[31.0, -99.0], zoom_start=6, tiles='CartoDB positron', control_scale=True)
m.fit_bounds([[25.8, -106.7], [36.6, -93.4]])

tooltip_fields = [c for c in ['COMID', 'GNIS_NAME', 'LENGTHKM', 'REACHCODE', 'flow_cms_txt', 'flow_acft_txt', 'matched'] if c in flow_map_gdf.columns]
aliases = ['Jan 2018 Flow (cms)' if c == 'flow_cms_txt' else 'Jan 2018 Flow (ac-ft)' if c == 'flow_acft_txt' else c for c in tooltip_fields]

## 7. Style Flowlines by Discharge Magnitude

In [ ]:
q90 = float(flow_map_gdf['flow_cms_jan_2018'].quantile(0.9)) if len(flow_map_gdf) else 0.0
q50 = float(flow_map_gdf['flow_cms_jan_2018'].quantile(0.5)) if len(flow_map_gdf) else 0.0

def style_fn(feat):
    v = feat['properties'].get('flow_cms_jan_2018', 0.0) or 0.0
    if v <= 0:
        return {'color': '#9aa0a6', 'weight': 1.1, 'opacity': 0.45}
    if v <= q50:
        return {'color': '#67a9cf', 'weight': 1.6, 'opacity': 0.8}
    if v <= q90:
        return {'color': '#2166ac', 'weight': 2.2, 'opacity': 0.9}
    return {'color': '#08306b', 'weight': 3.0, 'opacity': 0.95}

folium.GeoJson(
    data=flow_map_gdf[[c for c in list(dict.fromkeys(tooltip_fields + ['flow_cms_jan_2018', 'geometry'])) if c in flow_map_gdf.columns]].to_json(),
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=tooltip_fields, aliases=aliases, sticky=False, localize=True),
    popup=folium.GeoJsonPopup(fields=tooltip_fields, aliases=aliases, localize=True, labels=True),
    name='Flowlines (Jan 2018 routed discharge)'
).add_to(m)

def prep_points(df):
    x = df.copy()
    for col in ['CPID', 'ID', 'Gage_ID', 'Type']:
        if col not in x.columns:
            x[col] = ''
    for col in ['LAT', 'LONG', 'JAN', 'Year']:
        if col not in x.columns:
            x[col] = np.nan
    x['LAT'] = pd.to_numeric(x['LAT'], errors='coerce')
    x['LONG'] = pd.to_numeric(x['LONG'], errors='coerce')
    x['JAN'] = pd.to_numeric(x['JAN'], errors='coerce')
    x['Year'] = pd.to_numeric(x['Year'], errors='coerce')
    x = x[x['Year'] == 2018].copy() if (x['Year'] == 2018).any() else x
    return x.dropna(subset=['LAT', 'LONG']).drop_duplicates(subset=['CPID'])

usgs_plot = prep_points(usgs_df)
wam_plot = prep_points(wam_df)

usgs_fg = folium.FeatureGroup(name=f'USGS gages ({len(usgs_plot):,})', show=True)
for _, r in usgs_plot.iterrows():
    folium.CircleMarker(
        location=[float(r['LAT']), float(r['LONG'])],
        radius=4,
        color='#084081',
        fill=True,
        fill_color='#0868ac',
        fill_opacity=0.9,
        weight=1,
        popup=folium.Popup(f"USGS<br>CPID: {r.get('CPID','')}<br>ID: {r.get('ID','')}<br>Gage: {r.get('Gage_ID','')}<br>Jan ac-ft: {r.get('JAN', np.nan)}", max_width=320),
    ).add_to(usgs_fg)
usgs_fg.add_to(m)

wam_fg = folium.FeatureGroup(name=f'WAM points ({len(wam_plot):,})', show=True)
for _, r in wam_plot.iterrows():
    folium.CircleMarker(
        location=[float(r['LAT']), float(r['LONG'])],
        radius=3,
        color='#cb181d',
        fill=True,
        fill_color='#ef3b2c',
        fill_opacity=0.8,
        weight=1,
        popup=folium.Popup(f"WAM<br>CPID: {r.get('CPID','')}<br>ID: {r.get('ID','')}<br>Gage: {r.get('Gage_ID','')}<br>Jan ac-ft: {r.get('JAN', np.nan)}", max_width=320),
    ).add_to(wam_fg)
wam_fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

## 8. Add Join Diagnostics and Unmatched Reports

In [ ]:
matched_count = int(flow_map_gdf['matched'].sum())
total_count = len(flow_map_gdf)
match_rate = (matched_count / total_count * 100.0) if total_count else 0.0

unmatched = flow_map_gdf.loc[~flow_map_gdf['matched'], [c for c in ['COMID', 'REACHCODE', 'GNIS_NAME', 'LENGTHKM'] if c in flow_map_gdf.columns]].copy()
unmatched.to_csv(UNMATCHED_OUTPUT_PATH, index=False)

nonzero = flow_map_gdf.loc[flow_map_gdf['flow_cms_jan_2018'] > 0, 'flow_cms_jan_2018']
print(f'Match rate: {matched_count:,}/{total_count:,} ({match_rate:.2f}%)')
print('Unmatched sample:')
display(unmatched.head(10))
print(f'Unmatched CSV: {UNMATCHED_OUTPUT_PATH}')
if len(nonzero):
    print(f'Jan 2018 flow_cms stats -> min={nonzero.min():.6f}, median={nonzero.median():.6f}, max={nonzero.max():.6f}')
else:
    print('No positive Jan 2018 discharge after join.')

## 9. Save HTML Map and Run Cell-Level Validation Checks

In [ ]:
required_cols = {'flow_cms_jan_2018', 'discharge_jan_2018', 'matched'}
missing_cols = [c for c in required_cols if c not in flow_map_gdf.columns]
assert not missing_cols, f'Missing columns: {missing_cols}'
assert (flow_map_gdf['flow_cms_jan_2018'].notna().sum() > 0), 'No discharge values found after join.'

m.save(MAP_OUTPUT_PATH)
print(f'Saved interactive map HTML: {MAP_OUTPUT_PATH}')
print(f'Flowline features: {len(flow_map_gdf):,}')
print(f'Flowlines with routed discharge: {int(flow_map_gdf["matched"].sum()):,}')
m